In [1]:
# ruff: noqa: T201, T203, ERA001, BLE001
from __future__ import annotations

import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import json
import logging

import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

import src.prompts.system as system_prompts
from config.config import DATA_DIR, RUN_DIR
from config.runs_vars import DATASET_SUBSETS_MAP
from src.constants import PAIRS_SEPARATOR
from src.evaluate import (
    get_gt_pairs,
    get_predictions_with_gt,
    plot_usage_histograms,
    read_run_metrics_df,
    save_analysis_results,
    store_run_metrics_df,
)
from src.formatting import (
    format_gt_pairs_filepath,
    format_oracle_pairs_filepath,
    format_oracle_pairs_precomputed_dir,
    format_oracle_pairs_with_prompts_path,
    format_predictions_run_path,
    format_run_path,
    format_storing_pathes_from_run_path,
    format_subsets_ontologies_paths,
)
from src.LLM_servers.bedrock import BedrockServer
from src.LLM_servers.openai import OpenAIServer
from src.onto_access import OntologyAccess
from src.onto_object import OntologyEntryAttr
from src.processing import parallel_samples_process, save_oracle_pairs_with_prompts, try_load_precomputed_oracle_pairs
from src.prompts.prompts import (
    prompt_direct_entity,
    prompt_direct_entity_ontological,
    prompt_direct_entity_with_synonyms,
    prompt_sequential_hierarchy,
    prompt_sequential_hierarchy_ontological,
    prompt_sequential_hierarchy_with_synonyms,
)
from src.utils import read_oracle_pairs, save_run_results

pd.set_option("display.max_rows", None)
logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

In [16]:
for dataset in DATASET_SUBSETS_MAP:
    for subset in DATASET_SUBSETS_MAP[dataset]:
        oracle_pairs_dir = format_oracle_pairs_precomputed_dir(dataset, subset)
        reduced_oracle_pairs_dir = format_oracle_pairs_precomputed_dir(dataset, subset, suffix="_reduced")
        print(f"Processing dataset: {dataset}, subset: {subset}, {oracle_pairs_dir=}")

        reduced_pairs_path = format_oracle_pairs_filepath(dataset, subset)
        reduced_pairs_path = reduced_pairs_path.with_name(reduced_pairs_path.stem + "_reduced.txt")
        reduced_pairs_df = pd.read_csv(reduced_pairs_path, sep="|", header=None)

        reduced_pairs = set(zip(reduced_pairs_df[0], reduced_pairs_df[1]))
        print(f"Number of reduced pairs: {len(reduced_pairs)}")

        # copy oracle pairs to reduced dir
        reduced_oracle_pairs_dir.mkdir(parents=True, exist_ok=True)

        for file in oracle_pairs_dir.glob("*.csv"):
            target_file = reduced_oracle_pairs_dir / file.name
            whole_df = pd.read_csv(file, sep="|", header=None)

            whole_df["is_reduced"] = whole_df.apply(lambda row: (row[0], row[1]) in reduced_pairs, axis=1)
            reduced_prompts_df = whole_df[whole_df["is_reduced"]].drop(columns=["is_reduced"])

            print("Length of original file:", len(whole_df))
            print("Length of reduced file:", len(reduced_prompts_df))
            print(f"Saving reduced file to: {target_file}")

            reduced_prompts_df.to_csv(target_file, sep="|", index=False, header=False)

Processing dataset: bioml-2024, subset: ncit-doid, oracle_pairs_dir=PosixPath('data/bioml-2024/ncit-doid/oracle_pairs')
Number of reduced pairs: 856
Length of original file: 1248
Length of reduced file: 856
Saving reduced file to: data/bioml-2024/ncit-doid/oracle_pairs_reduced/oasystem_mappings-prompt_direct_entity.csv
Length of original file: 1248
Length of reduced file: 856
Saving reduced file to: data/bioml-2024/ncit-doid/oracle_pairs_reduced/oasystem_mappings-prompt_only_with_parents.csv
Length of original file: 1248
Length of reduced file: 856
Saving reduced file to: data/bioml-2024/ncit-doid/oracle_pairs_reduced/oasystem_mappings-prompt_sequential_hierarchy_with_synonyms.csv
Length of original file: 1248
Length of reduced file: 856
Saving reduced file to: data/bioml-2024/ncit-doid/oracle_pairs_reduced/oasystem_mappings-prompt_direct_entity_ontological.csv
Length of original file: 1248
Length of reduced file: 856
Saving reduced file to: data/bioml-2024/ncit-doid/oracle_pairs_reduc